In [ ]:
from typing import NamedTuple, Callable, Self
from jaxtyping import Array, Float, Scalar
from functools import partial
import warnings

import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import jax.random as jr


jax.config.update("jax_enable_x64", True)
EPS = float(jnp.sqrt(jnp.finfo(float).eps))  # Machine precision for float64
SEED = 42
np.random.seed(SEED)

# Problem Parameters

In [ ]:
X0, Y0 = -2.0, 1.0  # Starting coordinates
X1, Y1 = 0.0, 0.0  # Ending coordinates
GRAVITY: float = 9.81  # Gravitational acceleration

assert Y1 < Y0, "The endpoint must be below the start point."
assert X1 > X0, "The endpoint must be to the right of the start point."

In [ ]:
def find_brachistochrone(interp_points: int = 100000):
    # The cycloid can be parameterized implicitly as a function of the angle θ:
    #   Δx = r (θ - sin θ)
    #   Δy = -r (1 - cos θ)
    # the final angle θ1 is the one that satisfies:
    #   (θ - sin θ) / (1 - cos θ) - (X1 - X0) / (Y0 - Y1) = 0

    # find θ1 (final angle) using root finding with newton's method
    @jax.jit
    @jax.value_and_grad
    def f(angle):
        angle = angle % (2 * jnp.pi)
        return (angle - jnp.sin(angle)) / (1.0 - jnp.cos(angle)) - (X1 - X0) / (Y0 - Y1)

    final_angle = sp.optimize.root_scalar(f, fprime=True, x0=2.0).root % (2 * jnp.pi)

    # compute the radius r of the cycloid and the total travel time
    r = (Y0 - Y1) / (1.0 - np.cos(final_angle))
    omega = jnp.sqrt(GRAVITY / r)
    travel_time = final_angle / omega

    # compute a dense grid of interpolation points
    angles = omega * jnp.linspace(0.0, travel_time, interp_points)
    x_vals = X0 + r * (angles - jnp.sin(angles))
    y_vals = Y0 - r * (1.0 - jnp.cos(angles))

    # return cyclois as a callable and the optimal travel time
    cycloid = lambda x: jnp.interp(x, x_vals, y_vals)
    travel_time = jnp.sqrt(r / GRAVITY) * final_angle
    return cycloid, travel_time


def upper_bound(x, normalized=False):
    if not normalized:
        x = (x - X0) / (X1 - X0)
    return Y0 + (Y1 - Y0) * x


def lower_bound(x, normalized=False):
    if not normalized:
        x = (x - X0) / (X1 - X0)
    line = 2 * Y1 - Y0 - (Y1 - Y0) * x
    return line + 2 * (Y0 - Y1) * jnp.exp(-10 * jnp.sqrt(x))


def travel_time(f: Callable):
    @jax.jit
    def dtdx(x):
        lb, ub = lower_bound(x), upper_bound(x)
        x = jnp.array(x).reshape(1, 1)
        y = f(x).squeeze()
        y = jnp.clip(y, lb, ub)
        dydx = jax.jacobian(f)(x).squeeze()
        v = jnp.sqrt(2.0 * GRAVITY * (Y0 - y))
        return jnp.sqrt(1.0 + dydx**2) / v.clip(min=EPS)

    t, error = sp.integrate.quad(dtdx, X0, X1, limit=100, epsabs=1e-4, epsrel=1e-4)
    return jnp.array(t)


cycloid, optimal_time = find_brachistochrone()
print(f"cycloid  = {travel_time(cycloid):.8f} s")
print(f"optimal   = {optimal_time:.8f} s")

In [ ]:
class RKHSFunction(NamedTuple):
    # f = sum ai k(xi, .)
    x: Float[Array, "k d"]  # basis points, in this case d=1
    a: Float[Array, "k"]  # function values at the basis points

    @staticmethod
    @partial(jax.vmap, in_axes=(0, None))
    @partial(jax.vmap, in_axes=(None, 0))
    def kernel(a: Float[Array, "n d"], b: Float[Array, "m d"]) -> Float[Array, "n m"]:
        scale = 3.0
        d2 = jnp.sum(jnp.square(a - b))
        k = jnp.exp(-0.5 * d2 / scale)
        return k

    @jax.jit
    def __call__(self, t: Float[Array, "n d"]) -> Float[Array, "n"]:
        offset = upper_bound(t).squeeze()
        t = (t - X0) / (X1 - X0)  # [X0, X1] -> [0, 1]
        t = jsp.special.logit(t)  # [0, 1] -> (-inf, inf)
        Ktx = self.kernel(t, self.x)
        return Ktx @ self.a + offset.squeeze()

    @classmethod
    def sample(cls, n: int, k: int, d: int) -> list[Self]:
        lhs_sampler = sp.stats.qmc.LatinHypercube(
            d=k * (d + 1), seed=np.random.mtrand._rand
        )
        params = lhs_sampler.random(n=n).reshape(n, k, d + 1)
        return [cls.from_array(pi) for pi in params]

    @classmethod
    def from_array(cls, params: Float[Array, "k * (d+1)"], eps: float = 0.1) -> Self:
        x, a = params[:, :-1], params[:, -1]

        # ensure x is unique
        perm = jnp.argsort(x.flatten(), axis=0)
        x, a = x[perm], a[perm]
        x = x + jnp.arange(len(x))[:, None] * eps
        x = x / (1.0 + len(x) * eps)  # [0, 1+n*eps] -> [0, 1]
        x = x * (1 - 2 * eps) + eps  # [0, 1] -> [eps, 1-eps]

        # transform a into appropriate range
        lb = lower_bound(x.squeeze(), normalized=True)
        ub = upper_bound(x.squeeze(), normalized=True)
        a = -a * (ub - lb) / len(a) - EPS  # [0, 1] -> [-(ub-lb), -eps]

        # transform x into appropriate range
        x = jsp.special.logit(x)  # [0, 1] -> (-inf, inf)
        return cls(x=x, a=a)

In [ ]:
def plot_functions(fs: list[RKHSFunction], labels: list[str] | None = None):
    x_grid = np.linspace(X0, X1, 1000).reshape(-1, 1)
    plt.figure(figsize=(5, 3))
    plt.plot(x_grid, upper_bound(x_grid), "r:")
    plt.plot([X0, X1], [Y0, Y1], "ro", label="hard constraint")
    plt.plot(x_grid, lower_bound(x_grid), "r:", label="search space")
    plt.plot(x_grid, cycloid(x_grid), "k", label=f"cycloid (time={travel_time(cycloid):.4f} s)")
    for i, (fi, l) in enumerate(zip(fs, labels or [""] * len(fs))):
        x_basis = jsp.special.expit(fi.x) * (X1 - X0) + X0
        plt.plot(x_grid, fi(x_grid), label=f"{l} (time={travel_time(fi):.4f} s)")
        plt.plot(x_basis, fi(x_basis), "x", color=plt.gca().lines[-1].get_color())
    plt.grid()
    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.show()

# Fit GP

In [ ]:
class Gaussian(NamedTuple):
    mean: Float[Array, "n"]
    cov: Float[Array, "n n"]


class GaussianProcess(NamedTuple):
    # model parameters
    rho: Scalar  # lengthscale
    g: Scalar  # nugget
    nu: Scalar  # covariance scale
    b: Scalar  # trend

    # observed data
    observed_fs: list[RKHSFunction]
    observed_ys: Float[Array, "n"]

    # cached covariance matrix of the observed data
    Koo: Float[Array, "n n"]

    @classmethod
    def fit(
        cls,
        fs: list[RKHSFunction],
        ys: Float[Array, "n"],
        *,
        lengthscale_range: tuple[float, float] = (EPS, 100.0),
        nugget_range: tuple[float, float] = (EPS, 1e-3),
        max_iterations: int = 100,
    ):
        # define the loss function for optimization
        def loglikelihood(
            rho: Scalar, g: Scalar, fs: list[RKHSFunction], ys: Float[Array, "n"]
        ) -> tuple[Scalar, Scalar, Scalar]:
            # foward of kernel
            K = cls.kernel(rho, fs, fs)
            K = K + jnp.eye(len(ys)) * g

            # cholesky of K and compute logdet
            K_sqrt, is_lower = jsp.linalg.cho_factor(K)
            logdetK = 2.0 * jnp.sum(jnp.log(jnp.diag(K_sqrt)))

            # compute Ki_1=(K^-1 @ 1) and Ki_y=(K^-1 @ y)
            Ki_1, Ki_y = jsp.linalg.cho_solve(
                c_and_lower=(K_sqrt, is_lower),
                b=jnp.stack([jnp.ones_like(ys), ys], 1),
            ).T

            # compute optimal trend b and scale nu
            b = (Ki_1 * ys).sum() / Ki_1.sum()
            nu = jnp.dot((ys - b) / len(ys), (Ki_y - Ki_1 * b))

            # likelihood when marginalizing over trend and variance
            loglik = -0.5 * (len(ys) * jnp.log(nu) + logdetK)
            return loglik, b, nu

        @jax.jit
        @jax.value_and_grad
        def mle_loss(params: Float[Array, "2"]):
            rho, g = params[0], params[-1]
            loglik, b, nu = loglikelihood(rho, g, fs, ys)
            return -loglik

        # initialization
        nugget = min(0.1, nugget_range[1])
        lengthscale = 0.9 * lengthscale_range[1] + 0.1 * lengthscale_range[0]
        init_params = jnp.array([jnp.log(lengthscale), jnp.log(nugget)])

        # run optimization
        result = sp.optimize.minimize(
            fun=mle_loss,
            x0=init_params,
            jac=True,
            method="L-BFGS-B",
            bounds=[lengthscale_range, nugget_range],
            options=dict(maxiter=max_iterations, ftol=EPS, gtol=0),
        )

        # extract the optimal parameters and infer the rest
        rho = jnp.array(result.x[0])
        g = jnp.array(result.x[1])
        llk, b, nu = loglikelihood(rho, g, fs, ys)

        # precompute the covariance matrix of the observed data for faster predictions
        Koo = nu * (cls.kernel(rho, fs, fs) + g * jnp.eye(len(ys)))
        return cls(rho=rho, g=g, nu=nu, b=b, observed_fs=fs, observed_ys=ys, Koo=Koo)

    @staticmethod
    def kernel(
        rho: Scalar, fs_1: list[RKHSFunction], fs_2: list[RKHSFunction]
    ) -> Float[Array, "n m"]:
        def rkhs_dist_squared(f1: RKHSFunction, f2: RKHSFunction) -> Scalar:
            K11 = RKHSFunction.kernel(f1.x, f1.x)
            K12 = RKHSFunction.kernel(f1.x, f2.x)
            K22 = RKHSFunction.kernel(f2.x, f2.x)
            d2 = +f1.a @ K11 @ f1.a + f2.a @ K22 @ f2.a - 2 * f1.a @ K12 @ f2.a
            return jnp.maximum(d2, 0.0) # ensure non-negativity

        # stack list of pytrees into a single pytree of arrays
        d2 = jnp.array([[rkhs_dist_squared(f1, f2) for f2 in fs_2] for f1 in fs_1])
        k = jnp.exp(-0.5 * d2 / rho)
        return k

    @jax.jit
    def predict(self, fs: list[RKHSFunction]) -> Gaussian:
        n = len(self.observed_ys)

        # compute covariance matrices
        Kxx = self.nu * self.kernel(self.rho, fs, fs)
        Kxo = self.nu * self.kernel(self.rho, fs, self.observed_fs)
        Koo = self.Koo

        # posterior mean and covariance
        mean = self.b + Kxo @ jnp.linalg.solve(Koo, self.observed_ys - self.b)
        cov = Kxx - Kxo @ jnp.linalg.solve(Koo, Kxo.T)

        # Add correction based on the trend estimation correlation
        Kbx = jnp.ones((1, n)) @ jnp.linalg.solve(Koo, Kxo.T)
        cov = cov + (1 - Kbx).T @ (1 - Kbx) / jnp.linalg.inv(Koo).sum()
        return Gaussian(mean=mean, cov=cov)

    @jax.jit
    def log_expected_improvement(self, f: RKHSFunction) -> Scalar:
        # numerically stable version following https://arxiv.org/pdf/2310.20708:
        mu, cov = self.predict([f])
        mu = mu.squeeze()
        sigma = cov.squeeze() ** 0.5
        z = (self.observed_ys.min() - mu) / sigma

        # use lax.cond to avoid propagating NaNs in the gradients
        branch1 = lambda: jnp.log(z * jsp.stats.norm.cdf(z) + jsp.stats.norm.pdf(z))
        branch2a = lambda: -2 * jnp.log(-z)
        branch2b = lambda: jax.nn.log1mexp(
            -jnp.log(-z)
            - jsp.stats.norm.logsf(-z)
            - z**2 / 2
            - jnp.log(2 * jnp.pi) / 2.0
        )
        branch2 = lambda: (
            -(z**2) / 2
            - jnp.log(2 * jnp.pi) / 2
            + jax.lax.cond(z < -1 / jnp.sqrt(EPS), branch2a, branch2b)
        )
        ei = jnp.log(sigma) + jax.lax.cond(z > -1, branch1, branch2)
        return ei

# Expected Improvement

In [ ]:
class BFGS(NamedTuple):
    multi_starts: int
    raw_samples: int = 1024
    max_iterations: int = 100
    ftol: float = EPS
    gtol: float = 0.0

    def __call__(self, model: GaussianProcess, k: int, d: int) -> RKHSFunction:
        # define the acquisition function and its gradient as a vect
        @jax.jit
        @jax.value_and_grad
        def loss(x: Float[Array, "k*(d+1)"]):
            f = RKHSFunction.from_array(x.reshape(k, d + 1))
            return -model.log_expected_improvement(f)

        # sample initial guess for candidate points
        lhs_sampler = sp.stats.qmc.LatinHypercube(
            d=k * (d + 1), seed=np.random.mtrand._rand
        )
        x = lhs_sampler.random(n=self.raw_samples)  # (n, k * (d+1))

        # keep the most promising initial guesses
        losses = np.array([loss(xi)[0] for xi in x])
        x = x[np.argsort(losses)[: self.multi_starts]]

        # optimize each initial guess with L-BFGS
        results = [
            sp.optimize.minimize(
                fun=loss,
                x0=xi,
                jac=True,
                method="L-BFGS-B",
                bounds=[(0.0, 1.0)] * (k * (d + 1)),
                options=dict(
                    maxiter=self.max_iterations,
                    ftol=self.ftol,
                    gtol=self.gtol,
                ),
            )
            for xi in x
        ]

        # sort results and return best q
        losses = np.array([result.fun for result in results])
        fs = [RKHSFunction.from_array(result.x.reshape(k, d + 1)) for result in results]
        return fs[np.argmin(losses)]

# Initial model

In [ ]:
INITIAL_ACQUISITION = 4
ACQUISITIONS_EACH_K = 10

In [ ]:
fs = RKHSFunction.sample(n=INITIAL_ACQUISITION, k=1, d=1)
ys = jnp.array([travel_time(fi) for fi in fs])
models = [GaussianProcess.fit(fs, ys)]
plot_functions(models[-1].observed_fs)

# Acquisitions with 1 basis point

In [ ]:
k = 1
print(f"Acquisition with {k} basis points")
acquisition = BFGS(multi_starts=16*k)
for i in range(ACQUISITIONS_EACH_K):
    f = acquisition(models[-1], k=k, d=1)
    y = travel_time(f)

    fs = fs + [f]
    ys = jnp.concatenate([ys, y[None]])
    models.append(GaussianProcess.fit(fs, ys))

    best_idx = jnp.argmin(ys)
    print(f"Iteration {i+1}: best time = {ys[best_idx]:.8f} s, optimal time = {optimal_time:.8f} s")
    plot_functions([fs[best_idx], fs[-1]], labels=["best found", "latest candidate"])

In [ ]:
plt.figure(figsize=(5, 3))
n0, dn = INITIAL_ACQUISITION, ACQUISITIONS_EACH_K
plt.plot(range(0, n0), ys[:n0], "o", label="initial samples")
plt.plot(range(n0, n0 + dn), ys[n0 : n0 + dn], "o", label="acquired samples (k=1)")
plt.axhline(optimal_time, color="red", linestyle="--", label="optimal time")
plt.xlabel("Total Evaluations")
plt.ylabel("Travel Time (s)")
plt.title("Convergence of Bayesian Optimization")
plt.grid()
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

# Acquisitions with 2 basis points

In [ ]:
k = 2
print(f"Acquisition with {k} basis points")
acquisition = BFGS(multi_starts=16*k)
for i in range(ACQUISITIONS_EACH_K):
    f = acquisition(models[-1], k=k, d=1)
    y = travel_time(f)

    fs = fs + [f]
    ys = jnp.concatenate([ys, y[None]])
    models.append(GaussianProcess.fit(fs, ys))

    best_idx = jnp.argmin(ys)
    print(f"Iteration {i+1}: best time = {ys[best_idx]:.8f} s, optimal time = {optimal_time:.8f} s")
    plot_functions([fs[best_idx], fs[-1]], labels=["best found", "latest candidate"])

In [ ]:
plt.figure(figsize=(5, 3))
n0, dn = INITIAL_ACQUISITION, ACQUISITIONS_EACH_K
plt.plot(range(0, n0), ys[:n0], "o", label="initial samples")
plt.plot(range(n0, n0 + dn), ys[n0 : n0 + dn], "o", label="acquired samples (k=1)")
plt.plot(range(n0 + dn, n0 + 2*dn), ys[n0 + dn : n0 + 2*dn], "o", label="acquired samples (k=2)")
plt.axhline(optimal_time, color="red", linestyle="--", label="optimal time")
plt.xlabel("Total Evaluations")
plt.ylabel("Travel Time (s)")
plt.title("Convergence of Bayesian Optimization")
plt.grid()
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

# Acquisitions with 4 basis points

In [ ]:
k = 4
print(f"Acquisition with {k} basis points")
acquisition = BFGS(multi_starts=16*k)
for i in range(ACQUISITIONS_EACH_K):
    f = acquisition(models[-1], k=k, d=1)
    y = travel_time(f)

    fs = fs + [f]
    ys = jnp.concatenate([ys, y[None]])
    models.append(GaussianProcess.fit(fs, ys))

    best_idx = jnp.argmin(ys)
    print(f"Iteration {i+1}: best time = {ys[best_idx]:.8f} s, optimal time = {optimal_time:.8f} s")
    plot_functions([fs[best_idx], fs[-1]], labels=["best found", "latest candidate"])

In [ ]:
plt.figure(figsize=(5, 3))
n0, dn = INITIAL_ACQUISITION, ACQUISITIONS_EACH_K
plt.plot(range(0, n0), ys[:n0], "o", label="initial samples")
plt.plot(range(n0, n0 + dn), ys[n0 : n0 + dn], "o", label="acquired samples (k=1)")
plt.plot(range(n0 + dn, n0 + 2*dn), ys[n0 + dn : n0 + 2*dn], "o", label="acquired samples (k=2)")
plt.plot(range(n0 + 2*dn, n0 + 3*dn), ys[n0 + 2*dn : n0 + 3*dn], "o", label="acquired samples (k=4)")
plt.axhline(optimal_time, color="red", linestyle="--", label="optimal time")
plt.xlabel("Total Evaluations")
plt.ylabel("Travel Time (s)")
plt.title("Convergence of Bayesian Optimization")
plt.grid()
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()